In [ ]:
import torch
from torch import nn

Y = torch.tensor([
    [4, 7, 1, 0],
    [5, 8, 9, 1]
])

bos_id = 2

def make_decoder_input(Y, bos_id):
    batch_size = Y.shape[0]
    bos_tensor = torch.full((batch_size, 1), bos_id, dtype=Y.dtype, device=Y.device)
    
    decoder_input = torch.cat([bos_tensor, Y[:, :-1]], dim=1)
    
    return decoder_input



make_decoder_input(Y, bos_id)
        

tensor([[2, 4, 7, 1],
        [2, 5, 8, 9]])

In [ ]:
#Seq2Seq
class Seq2Seq(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        embed_dim,
        hidden_size
    ):
        super().__init__()
        self.src_embedding = nn.Embedding(src_vocab_size,embed_dim)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size,embed_dim)
        self.encoder = nn.LSTM(embed_dim,hidden_size,batch_first=True)
        self.decoder = nn.LSTM(embed_dim,hidden_size,batch_first=True)
        self.map = nn.Linear(hidden_size,tgt_vocab_size)
        # 1. 源语言 Embedding
        # 2. 目标语言 Embedding
        # 3. Encoder LSTM
        # 4. Decoder LSTM
        # 5. 将 Decoder 隐藏状态映射到目标词表的 Linear

    def forward(self, src, tgt_in):
        # src:    (B, T_src)
        # tgt_in: (B, T_tgt)
        src_vec = self.src_embedding(src)
        _,(h_n,c_n) = self.encoder(src_vec)
        tgt_vec = self.tgt_embedding(tgt_in)
        outputs,_ = self.decoder(tgt_vec,(h_n,c_n))
        logits = self.map(outputs)
        # 1. 将源语言 Token IDs 转成词向量
        # 2. Encoder 读取源语言并得到 (h, c)
        # 3. 将目标语言 Token IDs 转成词向量
        # 4. Decoder 以 (h, c) 为初始状态处理目标输入
        # 5. 将 Decoder 输出映射为词表 Logits

        return logits
    
    
src = torch.tensor([
    [3, 5, 7, 1],
    [4, 8, 1, 0]
])

Y = torch.tensor([
    [6, 9, 1, 0],
    [5, 7, 8, 1]
])

model = Seq2Seq(
    src_vocab_size=20,
    tgt_vocab_size=30,
    embed_dim=16,
    hidden_size=32
)

tgt_in = make_decoder_input(Y, bos_id=2)

logits = model(src, tgt_in)

print(logits.shape)

torch.Size([2, 4, 30])
